In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression

df = pd.read_csv('../data/processed/battery_summary.csv')

feature_cols = ["capacity_at_cycle_10", "capacity_at_cycle_100", 
                 "capacity_fade_10_to_100", "internal_resistance_at_100"]
target_col = "cycle_life"

X = df[feature_cols]
y = df[target_col]

kf = KFold(n_splits=5, shuffle=True, random_state=42)

In [2]:
model = LinearRegression()
scores = cross_val_score(model, X, y, cv=kf, scoring='neg_mean_absolute_error')
mae_scores = -scores

print("MAE per fold:", np.round(mae_scores, 1))
print(f"Average MAE: {mae_scores.mean():.1f} cycles")
print(f"Std deviation: {mae_scores.std():.1f} cycles")

MAE per fold: [260.4 155.  286.3 292.6 341.1]
Average MAE: 267.1 cycles
Std deviation: 61.8 cycles


In [3]:
from sklearn.dummy import DummyRegressor

dummy = DummyRegressor(strategy="mean")
dummy_scores = cross_val_score(dummy, X, y, cv=kf, scoring='neg_mean_absolute_error')
dummy_mae = -dummy_scores

print("Dummy (always predict average) MAE per fold:", np.round(dummy_mae, 1))
print(f"Dummy average MAE: {dummy_mae.mean():.1f} cycles")

Dummy (always predict average) MAE per fold: [234.9 222.6 228.5 164.2 317.5]
Dummy average MAE: 233.5 cycles


In [4]:
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=1.0)
ridge_scores = cross_val_score(ridge, X, y, cv=kf, scoring='neg_mean_absolute_error')
ridge_mae = -ridge_scores

print(f"Ridge average MAE: {ridge_mae.mean():.1f} cycles")

# Reduced feature set - drop the redundant capacity_at_cycle_10
X_reduced = df[["capacity_at_cycle_100", "capacity_fade_10_to_100", "internal_resistance_at_100"]]
reduced_scores = cross_val_score(LinearRegression(), X_reduced, y, cv=kf, scoring='neg_mean_absolute_error')
reduced_mae = -reduced_scores

print(f"Reduced-features linear regression MAE: {reduced_mae.mean():.1f} cycles")

Ridge average MAE: 233.6 cycles
Reduced-features linear regression MAE: 263.9 cycles


In [5]:
# Load the new curve-based feature and merge it into our existing df
df_curve = pd.read_csv('../data/processed/curve_features_v2.csv')
df = df.merge(df_curve[["cell_id", "Qdlin_variance_10_100"]], on="cell_id")

print(df.shape)
print(df.head())

(44, 7)
   cell_id  cycle_life  capacity_at_cycle_10  capacity_at_cycle_100  \
0        0        1009              1.070419               1.069454   
1        1        1063              1.068262               1.066478   
2        2        1267              1.062787               1.060614   
3        3        1115              1.065105               1.063595   
4        4        1048              1.074539               1.073859   

   capacity_fade_10_to_100  internal_resistance_at_100  Qdlin_variance_10_100  
0                 0.000965                    0.015271               0.000057  
1                 0.001784                    0.015059               0.000052  
2                 0.002174                    0.014477               0.000017  
3                 0.001511                    0.015063               0.000076  
4                 0.000680                    0.015180               0.000066  


In [6]:
# Updated feature set - original 4 plus the new curve-based feature
feature_cols_v2 = ["capacity_at_cycle_10", "capacity_at_cycle_100",
                    "capacity_fade_10_to_100", "internal_resistance_at_100",
                    "Qdlin_variance_10_100"]

X_v2 = df[feature_cols_v2]
y = df["cycle_life"]

# Linear Regression with the new feature set
lr_v2_scores = cross_val_score(LinearRegression(), X_v2, y, cv=kf, scoring='neg_mean_absolute_error')
lr_v2_mae = -lr_v2_scores
print(f"Linear Regression (v2, with curve feature) MAE: {lr_v2_mae.mean():.1f} cycles")

# Ridge with the new feature set
ridge_v2_scores = cross_val_score(Ridge(alpha=1.0), X_v2, y, cv=kf, scoring='neg_mean_absolute_error')
ridge_v2_mae = -ridge_v2_scores
print(f"Ridge (v2, with curve feature) MAE: {ridge_v2_mae.mean():.1f} cycles")

print(f"\nReminder - dummy baseline MAE: 233.5 cycles")

Linear Regression (v2, with curve feature) MAE: 242.3 cycles
Ridge (v2, with curve feature) MAE: 233.6 cycles

Reminder - dummy baseline MAE: 233.5 cycles


In [7]:
X_solo = df[["Qdlin_variance_10_100"]]

lr_solo_scores = cross_val_score(LinearRegression(), X_solo, y, cv=kf, scoring='neg_mean_absolute_error')
lr_solo_mae = -lr_solo_scores
print(f"Linear Regression (Qdlin_variance_10_100 ONLY) MAE: {lr_solo_mae.mean():.1f} cycles")

Linear Regression (Qdlin_variance_10_100 ONLY) MAE: 211.3 cycles


In [8]:
print("MAE per fold:", np.round(lr_solo_mae, 1))
print(f"Std deviation: {lr_solo_mae.std():.1f} cycles")

MAE per fold: [212.3 266.7 184.1 116.7 276.8]
Std deviation: 58.4 cycles


In [9]:
import joblib

# Train the FINAL model on ALL the data (not just cross-validation folds) -
# cross-validation was for evaluating how good the approach is; now that
# we've picked our final approach (single-feature linear regression), we
# retrain it one last time using every available cell, so it learns from
# as much data as possible before being deployed.
final_model = LinearRegression()
final_model.fit(df[["Qdlin_variance_10_100"]], df["cycle_life"])

# Save to a dedicated models folder
output_model_path = '../models/battery_model.pkl'
joblib.dump(final_model, output_model_path)
print(f"Model saved to {output_model_path}")

Model saved to ../models/battery_model.pkl
